# 05 · Explore — the cognee memory is a real AgensGraph graph

Measure it, break it down by type, query it with raw Cypher, and visualize it — all against demo 1's `cognee_wiki` graph.

> Run `01_search_modes/build.py` first.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # examples/demos
from _common import config
config.require_openai_key(); config.quiet()          # quiet cognee's verbose logs
config.configure("cognee_wiki")
import cognee
from cognee.modules.search.types import SearchType
from cognee.infrastructure.databases.graph import get_graph_engine
g = await get_graph_engine()
m = await g.get_graph_metrics(include_optional=False)
for k in ("num_nodes", "num_edges", "mean_degree", "edge_density", "num_selfloops"):
    print(f"  {k:14} {m.get(k)}")

  num_nodes      5664
  num_edges      12895
  mean_degree    4.5533192090395485
  edge_density   0.00040202359253395267
  num_selfloops  7


## Node-type breakdown + connectivity

`get_graph_data` returns every node and edge; `get_disconnected_nodes` and `get_degree_one_nodes` are connectivity checks.

In [2]:
from collections import Counter
nodes, edges = await g.get_graph_data()
for t, c in Counter(p.get("type") for _, p in nodes).most_common():
    print(f"  {t or '?':14} {c}")
print("  isolated nodes:    ", len(await g.get_disconnected_nodes()))
print("  degree-one Entities:", len(await g.get_degree_one_nodes("Entity") or []))

  Entity         4083
  EntityType     531
  TextSummary    350
  TextDocument   350
  DocumentChunk  350


  isolated nodes:     0


  degree-one Entities: 0


## Slice the graph by node type (`get_filtered_graph_data`)

In [3]:
fnodes, fedges = await g.get_filtered_graph_data([{"type": ["Entity"]}])
print(f"  Entity-only subgraph: {len(fnodes)} nodes, {len(fedges)} edges")

  Entity-only subgraph: 4083 nodes, 3486 edges


## Top entities by degree — raw AgensGraph Cypher

In [4]:
rows = await g.query('''
MATCH (n:"__Node__") WHERE n.name IS NOT NULL
OPTIONAL MATCH (n)-[r]-()
WITH n.id AS id, n.name AS name, count(r) AS degree
RETURN id, name, degree ORDER BY degree DESC LIMIT 12''')
for r in rows:
    print(f"  {str(r.get('name'))[:34]:34} {r.get('degree')}")
top = rows[0] if rows else None

  person                             736
  concept                            452
  date                               269
  place                              182
  event                              166
  location                           129
  organization                       82
  film                               67
  azerbaijan                         60
  country                            59
  armenia                            50
  angola                             49


## Traverse from a node (`get_neighbors` / `get_connections`)

In [5]:
if top:
    nbrs = await g.get_neighbors(str(top["id"]))
    print(f"'{top['name']}' has {len(nbrs or [])} neighbors; first few:")
    for nb in (nbrs or [])[:5]:
        print(f"   - {nb.get('name') or nb.get('id')}  ({nb.get('type')})")
    conns = await g.get_connections(top["id"])
    print(f"get_connections -> {len(conns or [])} (node, edge, node) connections")

'person' has 736 neighbors; first few:
   - ronald wayne  (Entity)
   - steve wozniak  (Entity)
   - steve jobs  (Entity)
   - alfred elton van vogt  (Entity)
   - heinrich "henry" vogt  (Entity)


get_connections -> 736 (node, edge, node) connections


## Visualize the knowledge graph

In [6]:
out = str(config.DATA_DIR / "wiki_graph.html")
await cognee.visualize_graph(out)
print("wrote", out)

wrote .data/wiki_graph.html
